In [ ]:
%pip install opendartreader

In [ ]:
from typing import List, Dict
import FinanceDataReader as fdr

def get_top_k_stocks_manual(k: int) -> List[Dict[str, str]]:
    print(f"Running FDR to get top {k} stocks...")
    df = fdr.StockListing('KRX')
    df_filtered = df[df['Market'].isin(['KOSPI', 'KOSDAQ'])]
    df_sorted = df_filtered.dropna(subset=['Marcap']).sort_values(by='Marcap', ascending=False)
    result = df_sorted.head(k)[['Name', 'Code']].to_dict(orient='records')
    return result

In [ ]:
import OpenDartReader
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

dart = OpenDartReader(os.getenv("DART_API_KEY"))

xml_text_list = dart.document_all('20220816001711')
xml_text = xml_text_list[0]


In [ ]:
from web.domain.entity.company import CompanyEntity
from web.domain.entity.disclosure import DisclosureListEntity, DisclosureFileEntity
from web.domain.entity.finance import FinancialAccountEntity, FinancialIndexEntity
from web.domain.enums import DisclosureFileType
import pandas as pd
import re


class DartETLService:
    def __init__(self, db_session):
        self.db = db_session

    def get_corp_code(self, stock_code: str):
        """종목코드(005930)로 DART 고유번호(00126380) 찾기"""
        try:
            # OpenDartReader는 내부적으로 corp_code 매핑을 캐싱합니다.
            return dart.find_corp_code(stock_code)
        except Exception as e:
            print(f"[Error] 고유번호 찾기 실패 ({stock_code}): {e}")
            return None

    # ---------------------------------------------------------
    # 2. 기본 정보 저장 (Company)
    # ---------------------------------------------------------
    def sync_company_info(self, stock_code: str):
        #print(f">>> [Company] {stock_code} 기본 정보 동기화 시작...")

        corp_code = self.get_corp_code(stock_code)
        if not corp_code:
            return None

        # DART API 호출 (기업개황)
        # 반환값: dict 형태 (status, message, list 등 포함할 수 있음, OpenDartReader는 주로 dict/info 반환)
        try:
            data = dart.company(corp_code)
        except Exception as e:
            print(f"API 호출 실패: {e}")
            return None

        if not data:
            return None

        # Entity 매핑 (API 한글 필드명 -> DB 영문 필드명)
        # API 응답 예시: {'corp_name': '삼성전자', 'ceo_nm': '한종희', ...}
        mapped_data = {
            "corp_code": data.get('corp_code'),
            "name": data.get('corp_name'),
            "stock_code": data.get('stock_code'),  # 혹은 입력받은 stock_code 사용
            "ceo_name": data.get('ceo_nm'),
            "induty_code": data.get('induty_code'),
            "market": "KOSPI" if "유가" in data.get('corp_cls', '') else "KOSDAQ",  # 간이 로직
            "homepage_url": data.get('hm_url'),
            "headquarters_addr": data.get('adres'),
            "corporate_reg_no": data.get('jurir_no'),
            "business_reg_no": data.get('bizr_no'),
            "phone_number": data.get('phn_no'),
            "founded_date": pd.to_datetime(data.get('est_dt'), format='%Y%m%d', errors='coerce'),
            # Overview/Description 생성은 LLM을 통해 할 예정
            "overview": None,
            "description": None
        }

        # DB Upsert (기존 데이터 있으면 업데이트, 없으면 생성)
        existing = self.db.query(CompanyEntity).filter_by(stock_code=stock_code).first()

        if existing:
            # Update fields
            for key, value in mapped_data.items():
                setattr(existing, key, value)
            #print(f" - 업데이트 완료: {mapped_data['name']}")
            target_company = existing
        else:
            # Insert new
            new_company = CompanyEntity(**mapped_data)
            self.db.add(new_company)
            #print(f" - 신규 저장 완료: {mapped_data['name']}")
            target_company = new_company

        self.db.commit()
        self.db.refresh(target_company)
        return target_company

    # ---------------------------------------------------------
    # 3. 재무 정보 저장 (FinancialAccount)
    # ---------------------------------------------------------
    def sync_financial_info(self, company_entity, year: int = 2023, reprt_code: str = "11011"):
        """
        1. 재무제표 원본 저장 (FinancialAccount)
        2. 주요 지표 계산 및 저장 (FinancialIndex)
        """
        #print(f">>> [Finance] {company_entity.name} {year}년 재무제표 및 지표 동기화...")

        try:
            df = dart.finstate_all(company_entity.corp_code, year, reprt_code)
        except Exception as e:
            print(f"API 호출 실패: {e}")
            return

        if df is None or df.empty:
            print(" - 데이터 없음")
            return

        # [1] 지표 계산을 위한 임시 저장소
        # (키: 표준화된 용어, 값: 금액)
        key_metrics = {}

        count_account = 0

        # DataFrame 순회하며 계정 저장 + 핵심 데이터 추출
        for _, row in df.iterrows():
            account_nm = row['account_nm']
            amount_str = row['thstrm_amount']
            account_id = row.get('account_id')

            # 금액 전처리
            try:
                amount = int(amount_str.replace(',', '')) if amount_str and amount_str != '-' else 0
            except:
                amount = 0

            # --- A. FinancialAccount 저장 (기존 로직) ---
            existing_account = self.db.query(FinancialAccountEntity).filter_by(
                company_id=company_entity.id,
                bsns_year=year,
                reprt_code=reprt_code,
                account_nm=account_nm
            ).first()

            if existing_account:
                existing_account.thstrm_amount = amount
                existing_account.account_id = account_id
            else:
                new_account = FinancialAccountEntity(
                    company_id=company_entity.id,
                    bsns_year=year,
                    reprt_code=reprt_code,
                    account_id=account_id,
                    account_nm=account_nm,
                    thstrm_amount=amount
                )
                self.db.add(new_account)
            count_account += 1

            # --- B. 핵심 데이터 추출 (지표 계산용) ---
            # 공백 제거 및 표준화하여 매핑
            clean_nm = account_nm.replace(" ", "").strip()

            if clean_nm in ["자산총계", "자산"]:
                key_metrics['total_assets'] = amount
            elif clean_nm in ["부채총계", "부채"]:
                key_metrics['total_liabilities'] = amount
            elif clean_nm in ["자본총계", "자본"]:
                key_metrics['total_equity'] = amount
            elif clean_nm in ["매출액", "수익(매출액)", "영업수익"]:
                key_metrics['revenue'] = amount
            elif clean_nm in ["영업이익", "영업이익(손실)"]:
                key_metrics['operating_income'] = amount
            elif clean_nm in ["당기순이익", "당기순이익(손실)", "법인세비용차감전계속영업이익"]:
                # 연결 기준인 경우 '당기순이익'이 여러 개일 수 있음 (지배/비지배 등).
                # 가장 일반적인 '당기순이익'을 우선시하거나 logic 보강 필요.
                # 여기서는 덮어쓰기 방식으로 마지막 값 사용 (보통 전체 순이익이 됨)
                key_metrics['net_income'] = amount

        # --- C. 지표(Financial Index) 계산 ---
        calculated_indices = []

        # 값 가져오기 (없으면 0)
        assets = key_metrics.get('total_assets', 0)
        liabilities = key_metrics.get('total_liabilities', 0)
        equity = key_metrics.get('total_equity', 0)
        revenue = key_metrics.get('revenue', 0)
        op_income = key_metrics.get('operating_income', 0)
        net_income = key_metrics.get('net_income', 0)

        # 1. 부채비율 (Debt Ratio) = 부채 / 자본 * 100
        if equity > 0:
            ratio = (liabilities / equity) * 100
            calculated_indices.append(("부채비율", round(ratio, 2)))

        # 2. 유보율 등은 자본잉여금 데이터 필요하므로 생략 (필요시 추가)

        # 3. 영업이익률 (Operating Margin) = 영업이익 / 매출 * 100
        if revenue > 0:
            ratio = (op_income / revenue) * 100
            calculated_indices.append(("영업이익률", round(ratio, 2)))

        # 4. 순이익률 (Net Profit Margin) = 순이익 / 매출 * 100
        if revenue > 0:
            ratio = (net_income / revenue) * 100
            calculated_indices.append(("순이익률", round(ratio, 2)))

        # 5. ROA (총자산이익률) = 순이익 / 자산 * 100
        if assets > 0:
            ratio = (net_income / assets) * 100
            calculated_indices.append(("ROA", round(ratio, 2)))

        # 6. ROE (자기자본이익률) = 순이익 / 자본 * 100
        if equity > 0:
            ratio = (net_income / equity) * 100
            calculated_indices.append(("ROE", round(ratio, 2)))

        # --- D. 지표 DB 저장 ---
        count_index = 0
        for idx_name, idx_value in calculated_indices:
            existing_idx = self.db.query(FinancialIndexEntity).filter_by(
                company_id=company_entity.id,
                bsns_year=year,
                reprt_code=reprt_code,
                index_nm=idx_name
            ).first()

            if existing_idx:
                existing_idx.index_value = idx_value
            else:
                new_idx = FinancialIndexEntity(
                    company_id=company_entity.id,
                    bsns_year=year,
                    reprt_code=reprt_code,
                    index_nm=idx_name,
                    index_value=idx_value
                )
                self.db.add(new_idx)
            count_index += 1

        self.db.commit()
        print(f" - 계정 {count_account}개, 지표 {count_index}개 저장 완료")

    # ---------------------------------------------------------
    # 4. 공시 목록 저장 (DisclosureList)
    # ---------------------------------------------------------
    def _detect_file_type(self, content: str) -> DisclosureFileType:
        """
        내용(raw_content)을 분석하여 파일 타입을 추론합니다.
        """
        if not content:
            return DisclosureFileType.ETC

        content_head = content[:500].lower().strip()

        # 1. XML / XBRL 감지
        if "<?xml" in content_head or "<dart-xml" in content_head or "<document>" in content_head:
            # Enum에 XBRL이 있다면 XBRL, 없다면 ETC/XML로 매핑
            # (질문하신 코드의 Enum에는 XBRL이 있다고 가정)
            return DisclosureFileType.XBRL

        # 2. HTML 감지
        if "<html" in content_head or "<!doctype html" in content_head or "<head>" in content_head:
            return DisclosureFileType.HTML

        # 3. 그 외 (단순 텍스트 등)
        return DisclosureFileType.ETC

    def sync_disclosure_list(self, company_entity, start_date: str = '2024-01-01', end_date: str = '2024-12-31'):
#        print(f">>> [Disclosure] {company_entity.name} 최근 공시 {start_date} ~ {end_date} 및 본문 동기화...")

        try:
            # DART API 호출 (공시검색)
            # 최신순으로 정렬되어 오므로 상위 count개만 자름
            df = dart.list(company_entity.corp_code, start=start_date, end=end_date, kind='A')
        except Exception as e:
            print(f"API 호출 실패: {e}")
            return

        if df is None or df.empty:
            print(" - 공시 내역 없음")
            return


        for _, row in df.iterrows():
            rcept_no = row['rcept_no']

            # 1. DisclosureList (목록) 저장/조회
            disclosure = self.db.query(DisclosureListEntity).filter_by(rcept_no=rcept_no).first()

            if not disclosure:
                disclosure = DisclosureListEntity(
                    company_id=company_entity.id,
                    rcept_no=rcept_no,
                    report_nm=row['report_nm'],
                    rcept_dt=pd.to_datetime(row['rcept_dt'], format='%Y%m%d'),
                    flr_nm=row['flr_nm'],
                    rpt_type=None
                    #row["rpt_type"]
                )
                self.db.add(disclosure)
                # [중요] flush를 해야 ID가 생성되어 file 저장 시 참조 가능
                self.db.flush()

            # 2. DisclosureFile (본문) 저장
            # 이미 파일이 있는지 확인 (중복 다운로드 방지)
            existing_file = self.db.query(DisclosureFileEntity).filter_by(disclosure_id=disclosure.id).first()

            if not existing_file:
                try:
                    raw_text = dart.document(rcept_no)
                    detected_type = self._detect_file_type(raw_text)
                    # clean_text = re.sub('<[^<]+?>', '', raw_text)
                    # clean_text = re.sub(r'\n+', '\n', clean_text).strip()
                    if raw_text:
                        new_file = DisclosureFileEntity(
                            disclosure_id=disclosure.id,
                            # Enum 매핑 (여기서는 HTML로 가정, 실제론 XML일 수 있음)
                            file_type=detected_type,
                            # DART 뷰어 URL 생성
                            file_url=f"http://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
                            raw_content=raw_text
                        )
                        self.db.add(new_file)
                except Exception as e:
                    print(f"     -> 본문 다운로드 실패 ({rcept_no}): {e}")

        self.db.commit()

In [ ]:
from web.config.database import SessionLocal
from tqdm import tqdm
db = SessionLocal()
etl_service = DartETLService(db)

# 처리할 대상 기업 리스트
target_stocks = get_top_k_stocks_manual(1000)  # 삼성전자, SK하이닉스, LG엔솔

try:
    for stock in tqdm(target_stocks):
        print(f"\n====== Processing {stock} ======")
        # 1. 회사 기본 정보 저장
        company = etl_service.sync_company_info(stock['Code'])
        if company:
            # 2. 재무 정보 저장 (2024년 사업보고서)
            etl_service.sync_financial_info(company, year=2024, reprt_code="11011")

            # 3. 공시 목록 저장
            etl_service.sync_disclosure_list(company)

except Exception as e:
    print(f"치명적 오류 발생: {e}")
finally:
    db.close()
    print("\n[System] DB 세션 종료.")